In [1]:
!apt-get update
!apt-get install -y openmpi-bin libopenmpi-dev

Hit:1 http://archive.ubuntu.com/ubuntu jammy InRelease
Get:2 http://archive.ubuntu.com/ubuntu jammy-updates InRelease [128 kB]
Hit:3 https://cli.github.com/packages stable InRelease
Get:4 http://security.ubuntu.com/ubuntu jammy-security InRelease [129 kB]
Hit:5 https://ppa.launchpadcontent.net/deadsnakes/ppa/ubuntu jammy InRelease
Hit:6 https://ppa.launchpadcontent.net/ubuntugis/ppa/ubuntu jammy InRelease
Get:7 http://archive.ubuntu.com/ubuntu jammy-backports InRelease [127 kB]
Get:8 https://r2u.stat.illinois.edu/ubuntu jammy InRelease [6,555 B]
Get:9 http://archive.ubuntu.com/ubuntu jammy-updates/main amd64 Packages [4,170 kB]
Get:10 https://cloud.r-project.org/bin/linux/ubuntu jammy-cran40/ InRelease [3,632 B]
Get:11 http://archive.ubuntu.com/ubuntu jammy-updates/universe amd64 Packages [1,615 kB]
Get:12 http://security.ubuntu.com/ubuntu jammy-security/restricted amd64 Packages [6,794 kB]
Get:13 https://r2u.stat.illinois.edu/ubuntu jammy/main amd64 Packages [2,941 kB]
Get:14 http://s

In [34]:
%%writefile lab5.c
#include <mpi.h>
#include <stdio.h>
#include <stdlib.h>

int main(int argc, char** argv) {

    MPI_Init(&argc, &argv);

    int rank, size;
    MPI_Comm_rank(MPI_COMM_WORLD, &rank);
    MPI_Comm_size(MPI_COMM_WORLD, &size);

    if (argc < 2) {
        if (rank == 0)
            printf("Usage: %s <matrix_size_N>\n", argv[0]);
        MPI_Finalize();
        return 0;
    }

    int N = atoi(argv[1]);
    int local_rows = N / size;

    double *local_A = (double*) malloc(local_rows * N * sizeof(double));
    double *local_y = (double*) malloc(local_rows * sizeof(double));
    double *x = (double*) malloc(N * sizeof(double));

    double *A = NULL;
    double *y = NULL;

    if (rank == 0) {

        A = (double*) malloc(N * N * sizeof(double));
        y = (double*) malloc(N * sizeof(double));

        for (int i = 0; i < N * N; i++)
            A[i] = 1.0;

        for (int i = 0; i < N; i++)
            x[i] = 1.0;
    }

    MPI_Barrier(MPI_COMM_WORLD);
    double start_time = MPI_Wtime();

    //1. Scatter matrix A to local_A
    MPI_Scatter(A, local_rows * N, MPI_DOUBLE, local_A, local_rows *N, MPI_DOUBLE, 0, MPI_COMM_WORLD);

    //2. Broadcast vector x
    MPI_Bcast(x, N, MPI_DOUBLE, 0, MPI_COMM_WORLD);

    //3. Compute local matrix-vector product
    for (int i = 0; i < local_rows; i++)
    {
        local_y[i] = 0.0;
        for (int j = 0; j < N; j++)
        {
            local_y[i] += local_A[i*N + j] * x[j];
        }
    }

    //4. Gather results into y
    MPI_Gather(local_y, local_rows, MPI_DOUBLE, y, local_rows, MPI_DOUBLE, 0, MPI_COMM_WORLD);

    MPI_Barrier(MPI_COMM_WORLD);
    double end_time = MPI_Wtime();

    if (rank == 0) {
        printf("N = %d, Processes = %d, Time = %f seconds\n", N, size, end_time - start_time);
        free(A);
        free(y);
    }

    free(local_A);
    free(local_y);
    free(x);

    MPI_Finalize();
    return 0;
}

Overwriting lab5.c


In [35]:
!mpicc -O3 lab5.c -o lab5

In [63]:
!mpirun --allow-run-as-root --oversubscribe -np 1 ./lab5 20000

N = 20000, Processes = 1, Time = 2.604498 seconds
